In [ ]:
class CombinedBCELoss(nn.Module):
    def __init__(self, lambda_br=0.5, mu=1.5, pos_weight=None, label_smooth=0.01):
        super().__init__()
        self.lambda_br = lambda_br
        self.mu = mu
        self.label_smooth = label_smooth

        if pos_weight is not None:
            self.bce = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor([pos_weight], device=DEVICE)
            )
        else:
            self.bce = nn.BCEWithLogitsLoss()

    def smooth_labels(self, y):
        return y * (1 - self.label_smooth) + 0.5 * self.label_smooth

    def forward(self, outputs, labels):
        y = labels.to(DEVICE)
        y_sm = self.smooth_labels(y)

        branch_logits = outputs["branch_logits"]
        fused_logit = outputs["fused_logit"]

        loss_br = 0.0
        for _, logit in branch_logits.items():
            loss_br = loss_br + self.bce(logit, y_sm)

        loss_fuse = self.bce(fused_logit, y_sm)
        loss = self.lambda_br * loss_br + self.mu * loss_fuse

        return loss, loss_br.detach(), loss_fuse.detach()

In [ ]:
def compute_loss(outputs, labels, criterion, br_w=0.5, mu=1.5):
    """
    outputs:
      {
        "branch_logits": {"ts": ..., "thk": ...},
        "fuse_logit": ...
      }
    labels: [B]
    """
    branch_losses = []
    for _, logit in outputs["branch_logits"].items():
        branch_losses.append(criterion(logit, labels))

    if len(branch_losses) > 0:
        branch_loss = torch.stack(branch_losses).mean()
    else:
        branch_loss = 0.0

    fuse_loss = criterion(outputs["fuse_logit"], labels)

    total_loss = br_w * branch_loss + mu * fuse_loss
    return total_loss